# Seq2Seq 모델을 활용한 챗봇 만들기

![](https://wikidocs.net/images/page/24996/%EC%9D%B8%EC%BD%94%EB%8D%94%EB%94%94%EC%BD%94%EB%8D%94%EB%AA%A8%EB%8D%B8.PNG)

In [ ]:
# '묻고' & '답하는' 형태를 구성.

# Q "나이는 어떻게 되나요?" <- 질문을 인코더 입력
# A "저는 18살입니다" <-  디코더 출력

# '학습' 시 질문에 대한 압축된 정보가 context vector 안에 함축적으로 담겨 decoder 에 전달될거다.

# 기본 import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import tensorflow as tf
from tensorflow import keras

import random
def set_seed(seed = 42):
  tf.keras.utils.set_random_seed(seed)
  tf.config.experimental.enable_op_determinism()

set_seed(42)

from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences


In [ ]:
base_path = r'/content/drive/MyDrive'

In [ ]:
out_path = os.path.join(base_path, 'out')

# 데이터 셋 준비

In [ ]:
"""
Korpora 라는 자연어 처리 데이터셋.  꽤 괜찮다.
GitHub : https://github.com/ko-nlp/Korpora <- 다양한 데이터셋
공식 : https://pypi.org/project/Korpora/  <- 사용법, 예제등.
"""
None

In [ ]:
!pip install Korpora

- 이 중 챗봇용 데이터셋인 `KoreanChatbotKorpus`를 다운로드 받습니다.
- `KoreanChatbotKorpus` 데이터셋을 활용하여 챗봇 모델을 학습합니다.
- text, pair로 구성되어 있습니다.
- 질의는 **text**, 답변은 **pair**입니다.

In [ ]:
# from Korpora import KoreanChatbotKorpus

# corpus = KoreanChatbotKorpus()


## 방법b 직접 다운로드

In [ ]:
corpus = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv')
corpus.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [ ]:
corpus.shape

(11823, 3)

In [ ]:
corpus = corpus.head(6000)  # 학습 한계상 6000개의 데이터만

# 데이터 전처리

In [ ]:
texts = corpus['Q'].tolist() # 질문들
pairs = corpus['A'].tolist() # 답변들

## 특수 문자는 제거
한글과 숫자를 제외한 특수문자 제거

In [ ]:
import re

In [ ]:
def clean_sentence(sentence):
  # 한글, 숫자, 띄어쓰기를 제외한 모든 문자를 제거합니다.
  sentence = re.sub(r'[^0-9ㄱ-ㅎㅏ-ㅣ가-힣 ]', r'', sentence)
  return sentence

In [ ]:
clean_sentence("12시 땡^^!???")

'12시 땡'

In [ ]:
clean_sentence('abcd가나다라^^#$#%$12시 좋아~요')

'가나다라12시 좋아요'

## 형태소 분석 Konlpy

In [ ]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 21.1 MB/s eta 0:00:00


In [ ]:
from konlpy.tag import Okt

In [ ]:
okt = Okt()

In [ ]:
# 형태소 변환에 활용할 함수
def process_morph(sentence):
  return ' '.join(okt.morphs(sentence))

In [ ]:
process_morph('한글은 홀소리와 닿소리 모두 소리틀을 본떠 만든 음소문자[1]로 한글 맞춤법에서는 닿소리 14개와 홀소리 10개, 모두 24개를 표준으로 삼는다. "나랏말이 중국과 달라" 문제를 느낀 조선의 세종대왕이 한국어는 물론 이웃나라 말까지 나타내도록 1443년 창제하여 1446년 반포하였다. ')

'한글 은 홀소리 와 닿소리 모두 소 리틀 을 본떠 만든 음소문자 [ 1 ] 로 한글 맞춤법 에서는 닿소리 14 개 와 홀소리 10 개 , 모두 24 개 를 표준 으로 삼는다 . " 나랏말 이 중국 과 달라 " 문제 를 느낀 조선 의 세종대왕 이 한국어 는 물론 이웃 나라 말 까지 나타내도록 1443년 창제 하여 1446년 반포 하였다 .'

# 데이터셋 구성

**Seq2Seq** 모델이 학습하기 위한 데이터셋을 구성할 때, 다음과 같이 **3가지 데이터셋**을 구성합니다.

- `question`: encoder input 데이터셋 (질의 전체)
- `answer_input`: decoder input 데이터셋 (답변의 시작). **START 토큰**을 문장 처음에 추가 합니다.
- `answer_output`: decoder output 데이터셋 (답변의 끝). **END 토큰**을 문장 마지막에 추가 합니다.

## 문장을 입력받아 3가지 데이터 셋 구성하는 함수

In [ ]:
def clean_and_morph(sentence, is_question=True):
  # 한글 문장 전처리
  # 특수 문자 제거
  sentence = clean_sentence(sentence)
  # 형태소 변환
  sentence = process_morph(sentence)

  if is_question:
    return sentence  # Question 인 경우 바로 리턴.
  else:
    # Answer 인 경우
    # '<START> ' 토큰은 decoder input 에,  ' <END>' 토큰은 decoder output 에 추가
    return ('<START> ' + sentence, sentence + ' <END>')

In [ ]:
clean_and_morph("12시 땡!")

'12시 땡'

In [ ]:
clean_and_morph("12시 땡!", False)

('<START> 12시 땡', '12시 땡 <END>')

## `text` 와 `pair` 에 대한 데이터셋 구성

In [ ]:
# text 와 pair 에 대한 데이터 셋 구성 함수.

# 매개변수
#  texts=  질의
#  pairs=  답변

def preprocess(texts, pairs):
    questions = []  # encoder 의 입력
    answer_in = []  # decoder 의 입력
    answer_out = [] # decoder 의 출력 (타겟)

    # 질의에 대한 전처리
    for text in texts:
      question = clean_and_morph(text, is_question=True)
      questions.append(question)

    # 답변에 대한 전처리
    for pair in pairs:
      in_, out_ = clean_and_morph(pair, is_question=False)
      answer_in.append(in_)
      answer_out.append(out_)


    return questions, answer_in, answer_out

In [ ]:
questions, answer_in, answer_out = preprocess(texts, pairs)

In [ ]:
questions[:5]

['12시 땡', '1 지망 학교 떨어졌어', '3 박 4일 놀러 가고 싶다', '3 박 4일 정도 놀러 가고 싶다', '심하네']

In [ ]:
answer_in[:5]

['<START> 하루 가 또 가네요',
 '<START> 위로 해 드립니다',
 '<START> 여행 은 언제나 좋죠',
 '<START> 여행 은 언제나 좋죠',
 '<START> 눈살 이 찌푸려지죠']

In [ ]:
answer_out[:5]

['하루 가 또 가네요 <END>',
 '위로 해 드립니다 <END>',
 '여행 은 언제나 좋죠 <END>',
 '여행 은 언제나 좋죠 <END>',
 '눈살 이 찌푸려지죠 <END>']

## 단어사전 구축을 위해 모든 데이터 셋 문장을 합치기

In [ ]:
all_sentences = questions + answer_in + answer_out

len(all_sentences)

18000

In [ ]:
# 모든 문장을 공백으로 join 한뒤 다시 공백으로 쪼개기
a = (' '.join(questions) + ' '.join(answer_in) + ' '.join(answer_out)).split()

len(set(a))

7338

# 토큰화 (Tokenizer)

## 토큰의 option을 정의

- filter는 '(빈 문자열)'로 지정합니다.
- lower는 False로 지정합니다.
- oov_token은 '&lt;OOV&gt;'로 지정합니다.

In [ ]:
tokenizer = Tokenizer(filters='', lower=False, oov_token='<OOV>')

## fit_on_text() 단어사전 구축

In [ ]:
tokenizer.fit_on_texts(all_sentences)

In [ ]:
tokenizer.word_index

{'<OOV>': 1,
 '<START>': 2,
 '<END>': 3,
 '이': 4,
 '거': 5,
 '을': 6,
 '예요': 7,
 '가': 8,
 '해보세요': 9,
 '도': 10,
 '에': 11,
 '요': 12,
 '잘': 13,
 '를': 14,
 '보세요': 15,
 '사람': 16,
 '은': 17,
 '수': 18,
 '는': 19,
 '안': 20,
 '것': 21,
 '봐요': 22,
 '하세요': 23,
 '더': 24,
 '좋은': 25,
 '생각': 26,
 '너무': 27,
 '저': 28,
 '나': 29,
 '의': 30,
 '하고': 31,
 '할': 32,
 '마음': 33,
 '있을': 34,
 '마세요': 35,
 '하는': 36,
 '게': 37,
 '많이': 38,
 '말': 39,
 '친구': 40,
 '만': 41,
 '시간': 42,
 '해': 43,
 '때': 44,
 '들': 45,
 '같아요': 46,
 '죠': 47,
 '있어요': 48,
 '한': 49,
 '내': 50,
 '이에요': 51,
 '못': 52,
 '좀': 53,
 '으로': 54,
 '드세요': 55,
 '하지': 56,
 '다': 57,
 '고': 58,
 '에서': 59,
 '싶어': 60,
 '일': 61,
 '될': 62,
 '자신': 63,
 '지금': 64,
 '오늘': 65,
 '하면': 66,
 '사랑': 67,
 '가봐요': 68,
 '뭐': 69,
 '같아': 70,
 '로': 71,
 '다른': 72,
 '싶다': 73,
 '있는': 74,
 '연락': 75,
 '좋죠': 76,
 '건': 77,
 '돼요': 78,
 '랑': 79,
 '혼자': 80,
 '좋을': 81,
 '같이': 82,
 '네': 83,
 '공부': 84,
 '곳': 85,
 '제': 86,
 '이네': 87,
 '해주세요': 88,
 '일이': 89,
 '당신': 90,
 '적': 91,
 '살': 92,
 '걸': 93,
 '준비': 

## 단어의 개수

In [ ]:
len(tokenizer.word_index)

7338

In [ ]:
VOCAB_SIZE = len(tokenizer.word_index) + 1   # <- +1 은 padding 용  padding  문자 인덱스는 0
VOCAB_SIZE  # -> 나중에 one-hot encoding 시 필요

7339

## texts_to_sequences()

In [ ]:
# 문장을 정수 인덱스 시퀀스로 변환
question_sequence = tokenizer.texts_to_sequences(questions)
answer_in_sequence = tokenizer.texts_to_sequences(answer_in)
answer_out_sequence = tokenizer.texts_to_sequences(answer_out)

In [ ]:
print(questions[0])
print(question_sequence[0])

12시 땡
[3958, 5168]


In [ ]:
print(answer_in[0])
print(answer_in_sequence[0])

<START> 하루 가 또 가네요
[2, 298, 8, 118, 2513]


## pad_sequences()

In [ ]:
MAX_LENGTH = 30
TRUNCATING = 'post'
PADDING = 'post'

In [ ]:
question_padded = pad_sequences(question_sequence, maxlen=MAX_LENGTH, truncating=TRUNCATING, padding=PADDING)
answer_in_padded = pad_sequences(answer_in_sequence, maxlen=MAX_LENGTH, truncating=TRUNCATING, padding=PADDING)
answer_out_padded = pad_sequences(answer_out_sequence, maxlen=MAX_LENGTH, truncating=TRUNCATING, padding=PADDING)

In [ ]:
print(question_sequence[0])
print(question_padded[0])

[3958, 5168]
[3958 5168    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


In [ ]:
question_padded.shape

(6000, 30)

# to_categorical() 단어별 원핫인코딩

In [ ]:
from tensorflow.keras.utils import to_categorical

In [ ]:
# to_categorical 사용
answer_in_one_hot = keras.utils.to_categorical(answer_in_padded, num_classes=VOCAB_SIZE)
answer_out_one_hot = keras.utils.to_categorical(answer_out_padded, num_classes=VOCAB_SIZE)

In [ ]:
answer_in_one_hot[0].shape, answer_out_one_hot[0].shape

((30, 7339), (30, 7339))

# Tokenizer.index_word[idx]
index -> word 변환

In [ ]:
def convert_index_to_text(indexs, end_token):

  words = []


  # 모든 index 에 대해
  for index in indexs:
    if index == end_token: break  # 끝 단어에는 예측 중지

    # 사전에 존재하는 단어의 경우 단어를 추가
    if index > 0 and tokenizer.index_word[index] is not None:
      words.append(tokenizer.index_word[index])


  return ' '.join(words)


# 모델 생성

In [ ]:
# 모델 생성.